# In This project I will build my own LLM from scratch . I want to build a cs_ tutor LLM.

# Mini GPT From Scratch — TinyStories

## Introduction

In this project, I implemented and trained a **GPT-style language model from scratch** using the **TinyStories dataset**. The goal of this project was to understand the complete pipeline behind modern large language models, including tokenization, dataset preparation, transformer architecture, training, checkpointing, and text generation.

Instead of relying on a pretrained model, this project builds the entire system using **PyTorch** and trains it directly on text data. This allows a deeper understanding of how transformer-based language models learn to predict the next token in a sequence.

The workflow of this project includes:

* Loading and preparing the TinyStories dataset
* Training and loading a tokenizer
* Building a GPT-style transformer architecture
* Training the model with validation monitoring
* Saving checkpoints during training
* Loading the best checkpoint
* Generating text from prompts

Through this project, I explored how language models learn patterns in text and the challenges involved when training models from scratch with limited compute resources.


In [ ]:
import torch
print (torch.__version__)
print (torch.cuda.is_available())
print("GPU Namem: ", torch.cuda.get_device_name(0))

# Data Setup

### Install required Library

In [ ]:
# Install the library we need
#!pip install -q transformers datasets accelerate sentencepiece tokenizers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# create project folder
import os
PROJECT_PATH  = "/content/drive/MyDrive/mini_llm_project"
os.makedirs( PROJECT_PATH, exist_ok = True)
print("Project folder ready : ", PROJECT_PATH)

### Load TiniStories Dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("eminorhan/tinystories", "10M_1")
print(dataset)

#

### Check the data

In [ ]:
print(dataset["train"][0])

# Build the tokenizer

### Extract text from dataset

In [ ]:
# collecting training text
def get_training_corpus():
  for i in range(0, len(dataset["train"]), 1000 ):
    yield dataset["train"][i: i+1000]["text"]

### Import tokenizer tools

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

### Create Empty Tokenizer

In [ ]:
# Intialize tokenizer model
tokenizer = Tokenizer(models.BPE())

### Add pre_tokenizer

In [ ]:
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

### Train the tokenizer

In [ ]:
trainer = trainers.BpeTrainer(vocab_size = 16000,
                              special_tokens= [ "[PAD]", "[UNK]", "[BOS]", "[EOS]"])
tokenizer.train_from_iterator(get_training_corpus(), trainer = trainer)

### Test the Tokenizer

In [ ]:
encoded = tokenizer.encode("Hello, I want to learn computer science.")
print(encoded.tokens)
print(encoded.ids)

In [ ]:
print("Vocabulary size: ", tokenizer.get_vocab_size())

# Build a mini model from scratch

In [ ]:
# import + Reproducibility
import math, os, random
import torch.nn as nn
import torch.nn.functional as F
# make result more repeatable
torch.manual_seed(42)
random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: ", device)

### Define model settings

In [ ]:
# ===== Model size (small but real) =====
vocab_size = tokenizer.get_vocab_size()   # should be ~16000
block_size = 128      # max tokens the model reads at once
n_embd = 256          # how big each token's "meaning vector" is
n_head = 8            # number of attention heads
n_layer = 4           # number of transformer blocks
dropout = 0.1

print("vocab_size:", vocab_size)

### Build attention

In [ ]:
#  One Attention Head
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # Batch, Time, Channels(embedding)

        k = self.key(x)      # (B, T, head_size)
        q = self.query(x)    # (B, T, head_size)

        # attention scores: (B, T, T)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)

        # mask future tokens (so it can't cheat)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # softmax -> probabilities
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)    # (B, T, head_size)
        out = wei @ v        # (B, T, head_size)
        return out

In [ ]:
# Multi_head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
# FeedForward
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# Transformer Block
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # attention + skip connection
        x = x + self.ffwd(self.ln2(x)) # feedforward + skip connection
        return x

In [ ]:
# Mini Model
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                         # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)
        x = tok_emb + pos_emb                                             # (B, T, n_embd)

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                          # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))

        return logits, loss

In [ ]:
# quick test
model = MiniGPT().to(device)
print("Model parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")

# dummy batch (random tokens)
x = torch.randint(0, vocab_size, (2, 32)).to(device)
logits, loss = model(x, x)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

### Tokenize dataset

In [ ]:
# create a Tokeniztion Function
def tokenize_function(example):
    tokens = tokenizer.encode(example["text"]).ids
    return {"ids": tokens}

In [ ]:
# apply it to whole dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=["text"]
)

### Merging all tokens

In [ ]:
def group_texts(examples):
    # 1) Concatenate all token lists into one long list
    concatenated = sum(examples["ids"], [])

    # 2) We need at least block_size+1 so we can make (input, next_token_label)
    total_length = (len(concatenated) // (block_size + 1)) * (block_size + 1)
    concatenated = concatenated[:total_length]

    input_ids = []
    labels = []

    # 3) Make chunks of length block_size+1
    for i in range(0, total_length, block_size + 1):
        chunk = concatenated[i : i + block_size + 1]

        # inputs are first block_size tokens
        x = chunk[:-1]   # length = block_size

        # labels are next-token targets (shifted)
        y = chunk[1:]    # length = block_size

        input_ids.append(x)
        labels.append(y)

    return {"input_ids": input_ids, "labels": labels}

In [ ]:
lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True,
    remove_columns=tokenized_dataset["train"].column_names,
)

### Convert to PyTorch Format

In [ ]:
lm_dataset.set_format(type="torch")

In [ ]:
batch = lm_dataset["train"][0]
print(len(batch["input_ids"]), len(batch["labels"]))
print(batch["input_ids"][:10])
print(batch["labels"][:10])

### create DataLoader

In [ ]:
from torch.utils.data import DataLoader

batch_size = 16

train_loader = DataLoader(lm_dataset["train"], batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(lm_dataset["validation"], batch_size=batch_size, shuffle=False)

# quick check
batch = next(iter(train_loader))
print(batch["input_ids"].shape, batch["labels"].shape)

### Create Optimizer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

### Training

In [ ]:


# model.train()
# print_every = 200

# for step, batch in enumerate(train_loader):
#     input_ids = batch["input_ids"].to(device)
#     labels    = batch["labels"].to(device)

#     logits, loss = model(input_ids, labels)

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     if step % print_every == 0:
#         print(f"Step {step}, Loss: {loss.item():.4f}")

#     if step >= 2000:   # stop early for first test run
#         break

### Improving the model

In [ ]:
# Add a validation function
@torch.no_grad()
def estimate_loss(model, train_loader, val_loader, eval_iters=50):
    model.eval()
    out = {}
    for split, loader in [("train", train_loader), ("val", val_loader)]:
        losses = []
        for i, batch in enumerate(loader):
            if i >= eval_iters:
                break
            x = batch["input_ids"].to(device)
            y = batch["labels"].to(device)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

In [ ]:
#Upgrade  optimizer + scheduler + clipping
from torch.optim.lr_scheduler import CosineAnnealingLR

lr = 3e-4
weight_decay = 0.1
max_steps = 10000          # stronger than 2000
eval_every = 500
save_every = 1000
grad_clip = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=max_steps)

### Retrain

In [ ]:
# import os, torch

# CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints")
# os.makedirs(CKPT_DIR, exist_ok=True)

# model.train()
# step = 0

# while step < max_steps:
#     for batch in train_loader:
#         x = batch["input_ids"].to(device)
#         y = batch["labels"].to(device)

#         _, loss = model(x, y)

#         optimizer.zero_grad()
#         loss.backward()

#         # stabilize training
#         torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

#         optimizer.step()
#         scheduler.step()

#         if step % 200 == 0:
#             print(f"step {step} | loss {loss.item():.4f} | lr {scheduler.get_last_lr()[0]:.2e}")

#         # validation check
#         if step % eval_every == 0 and step > 0:
#             losses = estimate_loss(model, train_loader, val_loader, eval_iters=50)
#             print(f"[eval] step {step} | train {losses['train']:.4f} | val {losses['val']:.4f}")

#         # save checkpoint
#         if step % save_every == 0 and step > 0:
#             ckpt_path = os.path.join(CKPT_DIR, f"minigpt_step{step}.pt")
#             torch.save({
#                 "step": step,
#                 "model_state": model.state_dict(),
#                 "optimizer_state": optimizer.state_dict(),
#                 "vocab_size": vocab_size,
#                 "block_size": block_size,
#                 "n_embd": n_embd,
#                 "n_head": n_head,
#                 "n_layer": n_layer
#             }, ckpt_path)
#             print("Saved:", ckpt_path)

#         step += 1
#         if step >= max_steps:
#             break

## Load the best check point

In [ ]:
import torch, os

ckpt_path = os.path.join(PROJECT_PATH, "checkpoints", "minigpt_step9000.pt")

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.to(device)
model.eval()

print("Loaded checkpoint:", ckpt_path)
print("Step:", ckpt["step"])

In [ ]:
# # Save a “final model” file
# final_path = os.path.join(PROJECT_PATH, "final_minigpt.pt")
# torch.save(model.state_dict(), final_path)
# print("Saved final weights:", final_path)

# # tokenizer already saved earlier as tokenizer.json
# print("Tokenizer file:", os.path.join(PROJECT_PATH, "tokenizer.json"))

### Generation part

In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(model, start_text, max_new_tokens=80, temperature=0.9, top_k=50):
    model.eval()

    ids = tokenizer.encode(start_text).ids
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]  # keep last block_size tokens
        logits, _ = model(x_cond)

        logits = logits[:, -1, :]  # last position
        logits = logits / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

In [ ]:
print(generate(model, "Once upon a time", max_new_tokens=120))

In [ ]:
print(generate(model, "The little dog", max_new_tokens=120))
print(generate(model, "In a small village", max_new_tokens=120))

## Evaluate the scratch model

In [ ]:
@torch.no_grad()
def eval_scratch(model, loader, eval_iters=100):
    model.eval()
    losses = []

    for i, batch in enumerate(loader):
        if i >= eval_iters:
            break

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        _, loss = model(x, y)
        losses.append(loss.item())

    return sum(losses) / len(losses)


scratch_val_loss = eval_scratch(model, val_loader)

print("Scratch Model Validation Loss:", scratch_val_loss)
print("Scratch Perplexity:", torch.exp(torch.tensor(scratch_val_loss)).item())

# STEP 2
we built our model function  from sratch now we will try the use the build fonction and compare them to see the difference

### Build the “built” model

In [ ]:
# ===== Model size (small but real) =====
vocab_size = tokenizer.get_vocab_size()   # should be ~16000
block_size = 256     # max tokens the model reads at once
n_embd = 768        # how big each token's "meaning vector" is
n_head = 12           # number of attention heads
n_layer = 12           # number of transformer blocks
dropout = 0.1

print("vocab_size:", vocab_size)

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

built_config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

# enable weight tying
built_config.tie_word_embeddings = True
built_model = GPT2LMHeadModel(built_config).to(device)
built_model.train()

print("Built model params (M):", sum(p.numel() for p in built_model.parameters())/1e6)

### Train the built model

In [ ]:
# import time
# from torch.optim import AdamW

# built_optimizer = AdamW(built_model.parameters(), lr=3e-4, weight_decay=0.1)

# max_steps = 2000
# print_every = 200

# t0 = time.time()

# for step, batch in enumerate(train_loader):
#     if step >= max_steps:
#         break

#     x = batch["input_ids"].to(device)
#     y = batch["labels"].to(device)

#     out = built_model(input_ids=x, labels=y)
#     loss = out.loss

#     built_optimizer.zero_grad()
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(built_model.parameters(), 1.0)
#     built_optimizer.step()

#     if step % print_every == 0:
#         steps_per_sec = (step + 1) / (time.time() - t0)
#         print(f"[built] step {step} | loss {loss.item():.4f} | {steps_per_sec:.2f} steps/s")

### Evaluate built model

In [ ]:
# @torch.no_grad()
# def eval_built(model, loader, eval_iters=100):
#     model.eval()
#     losses = []

#     for i, batch in enumerate(loader):
#         if i >= eval_iters:
#             break

#         x = batch["input_ids"].to(device)
#         y = batch["labels"].to(device)

#         out = model(input_ids=x, labels=y)
#         losses.append(out.loss.item())

#     return sum(losses) / len(losses)

# built_val_loss = eval_built(built_model, val_loader)

# print("Built Model Validation Loss:", built_val_loss)
# print("Built Perplexity:", torch.exp(torch.tensor(built_val_loss)).item())

In [ ]:
# # continue training from where you stopped
# import time
# from torch.optim import AdamW

# # only create optimizer if you didn't already
# # if you already have built_optimizer, skip this line
# built_optimizer = AdamW(built_model.parameters(), lr=3e-4, weight_decay=0.1)

# max_steps = 10000
# print_every = 200

# t0 = time.time()

# for step, batch in enumerate(train_loader):
#     # IMPORTANT: step restarts at 0 in enumerate, so we just run extra steps.
#     # If you want exact 10k total steps, set this to 8000 more steps instead.
#     if step >= 8000:   # 2000 already done, do 8000 more
#         break

#     x = batch["input_ids"].to(device)
#     y = batch["labels"].to(device)

#     out = built_model(input_ids=x, labels=y)
#     loss = out.loss

#     built_optimizer.zero_grad()
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(built_model.parameters(), 1.0)
#     built_optimizer.step()

#     if step % print_every == 0:
#         steps_per_sec = (step + 1) / (time.time() - t0)
#         print(f"[built-continue] +{step} steps | loss {loss.item():.4f} | {steps_per_sec:.2f} steps/s")

In [ ]:
# built_val_loss = eval_built(built_model, val_loader)
# print("Built Model Validation Loss:", built_val_loss)
# print("Built Perplexity:", torch.exp(torch.tensor(built_val_loss)).item())

### Note
my scratch method beat the build method but this was possible because I train my scratch one very good than the build one .
For the net part I will continue with the build one to make my life easier

### Load TinyStories 100M_1



In [ ]:
from datasets import load_dataset

dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")
print(dataset_100m)
print(dataset_100m["train"][0]["text"][:200])

### Tokenize the 100M dataset

In [ ]:
EOS_ID = tokenizer.token_to_id("[EOS]")
assert EOS_ID is not None, "Tokenizer does not have [EOS] token"

def tokenize_with_eos(examples):
    ids_list = []
    for text in examples["text"]:
        ids = tokenizer.encode(text).ids
        ids.append(EOS_ID)          # end-of-story
        ids_list.append(ids)
    return {"ids": ids_list}

tok_100m = dataset_100m.map(
    tokenize_with_eos,
    batched=True,
    remove_columns=["text"]
)

In [ ]:
# def tokenize_batch(batch):
#     # batch["text"] is a list of strings
#     enc = [tokenizer.encode(t).ids for t in batch["text"]]
#     return {"input_ids": enc}

# tokenized_100m = dataset_100m.map(
#     tokenize_batch,
#     batched=True,
#     remove_columns=["text"],
# )
# print(tokenized_100m)

In [ ]:
block_size = 256   # keep your choice here

def make_blocks(examples):
    input_ids = []
    labels = []

    for ids in examples["ids"]:
        # slide through ONE story at a time
        for i in range(0, len(ids) - 1, block_size):
            chunk = ids[i : i + block_size + 1]

            # need at least 2 tokens to make x/y
            if len(chunk) < 2:
                continue

            x = chunk[:-1]
            y = chunk[1:]

            input_ids.append(x)
            labels.append(y)

    return {"input_ids": input_ids, "labels": labels}

lm_100m = tok_100m.map(
    make_blocks,
    batched=True,
    remove_columns=["ids"]
)

### Group tokens into fixed blocks

### DataLoader

In [ ]:
from torch.utils.data import DataLoader

batch_size = 8

train_loader_100m = DataLoader(lm_100m["train"], batch_size=batch_size, shuffle=True)
val_loader_100m = DataLoader(lm_100m["validation"], batch_size=batch_size, shuffle=False)

### build the model for 100M

### Evaluation

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

built_config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

# enable weight tying
built_config.tie_word_embeddings = True
built_model_100m = GPT2LMHeadModel(built_config).to(device)
built_model_100m.train()

print("Built model params (M):", sum(p.numel() for p in built_model.parameters())/1e6)

In [ ]:
import torch, math
from contextlib import nullcontext

@torch.no_grad()
def eval_built(model, loader, eval_iters=100):
    model.eval()
    losses = []
    for i, batch in enumerate(loader):
        if i >= eval_iters:
            break
        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)
        out = model(input_ids=x, labels=y)
        losses.append(out.loss.item())
    model.train()
    avg = sum(losses) / len(losses)
    ppl = math.exp(avg)
    return avg, ppl

### Training

In [ ]:
# import os, time
# import torch
# from torch.optim import AdamW

# PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
# CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints_built_100m")
# os.makedirs(CKPT_DIR, exist_ok=True)

# model = built_model_100m  # use the model you created
# model.train()

# lr = 3e-4
# weight_decay = 0.1
# max_steps = 10000          # start with 10k, we can extend later
# print_every = 200
# eval_every = 1000
# eval_iters = 100

# grad_accum_steps = 4       # effective batch = batch_size * 4
# use_amp = True

# optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# best_val = float("inf")
# best_path = None

# t0 = time.time()
# step = 0
# optimizer.zero_grad()

# train_iter = iter(train_loader_100m)

# while step < max_steps:
#     # get batch (loop forever)
#     try:
#         batch = next(train_iter)
#     except StopIteration:
#         train_iter = iter(train_loader_100m)
#         batch = next(train_iter)

#     x = batch["input_ids"].to(device)
#     y = batch["labels"].to(device)

#     autocast_ctx = torch.cuda.amp.autocast(enabled=use_amp)
#     with autocast_ctx:
#         out = model(input_ids=x, labels=y)
#         loss = out.loss / grad_accum_steps  # IMPORTANT

#     scaler.scale(loss).backward()

#     # update weights every grad_accum_steps
#     if (step + 1) % grad_accum_steps == 0:
#         scaler.unscale_(optimizer)
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         scaler.step(optimizer)
#         scaler.update()
#         optimizer.zero_grad()

#     # logging
#     if step % print_every == 0:
#         steps_per_sec = (step + 1) / (time.time() - t0)
#         print(f"[100M built] step {step} | train_loss {loss.item()*grad_accum_steps:.4f} | {steps_per_sec:.2f} steps/s")

#     # eval + save best
#     if step % eval_every == 0 and step > 0:
#         val_loss, val_ppl = eval_built(model, val_loader_100m, eval_iters=eval_iters)
#         print(f"[EVAL] step {step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")

#         ckpt_path = os.path.join(CKPT_DIR, f"built100m_step{step}.pt")
#         torch.save({
#             "step": step,
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "best_val": best_val,
#             "vocab_size": vocab_size,
#             "block_size": block_size,
#             "n_embd": n_embd,
#             "n_head": n_head,
#             "n_layer": n_layer,
#         }, ckpt_path)
#         print("Saved:", ckpt_path)

#         if val_loss < best_val:
#             best_val = val_loss
#             best_path = os.path.join(CKPT_DIR, "BEST.pt")
#             torch.save({
#                 "step": step,
#                 "model_state": model.state_dict(),
#                 "optimizer_state": optimizer.state_dict(),
#                 "best_val": best_val,
#             }, best_path)
#             print("✅ New BEST saved:", best_path)

#     step += 1

# print("Done. Best val loss:", best_val)
# print("Best checkpoint:", best_path)

In [ ]:
import torch, os

best_path = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"

ckpt = torch.load(best_path, map_location=device)
built_model_100m.load_state_dict(ckpt["model_state"])
built_model_100m.to(device)
built_model_100m.eval()

print("Loaded BEST checkpoint at step:", ckpt["step"])
print("Best val loss:", ckpt["best_val"])

In [ ]:
lens = [len(lm_100m["train"][i]["input_ids"]) for i in range(20)]
print(lens)

In [ ]:
import torch

PAD_ID = tokenizer.token_to_id("[PAD]")
if PAD_ID is None:
    PAD_ID = 0  # fallback

def collate_pad(batch):
    input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    labels    = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=PAD_ID)
    labels    = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)  # ignore padding in loss

    return {"input_ids": input_ids, "labels": labels}

In [ ]:
from torch.utils.data import DataLoader

batch_size = 8  # or 8 if GPU allows
train_loader_100m = DataLoader(lm_100m["train"], batch_size=batch_size, shuffle=True, collate_fn=collate_pad)
val_loader_100m   = DataLoader(lm_100m["validation"], batch_size=batch_size, shuffle=False, collate_fn=collate_pad)

In [ ]:
import time, math
import torch
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints_built_100m")
os.makedirs(CKPT_DIR, exist_ok=True)

model = built_model_100m

# ====== SETTINGS (LLM style) ======
target_steps = 50000            # train longer
print_every = 200
eval_every  = 2000
eval_iters  = 100

lr = 3e-4
weight_decay = 0.1
grad_accum_steps = 4
max_grad_norm = 1.0
use_amp = True

# ====== OPTIMIZER ======
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# If BEST checkpoint stored optimizer state, load it (so resume is real)
if "optimizer_state" in ckpt:
    optimizer.load_state_dict(ckpt["optimizer_state"])
    print("✅ Optimizer state loaded")

# ====== SCHEDULER (Warmup + Cosine) ======
# Scheduler steps happen only when optimizer.step() happens
# total optimizer updates = total_steps / grad_accum_steps
start_step = int(ckpt.get("step", 0))
total_updates = target_steps // grad_accum_steps
done_updates  = start_step // grad_accum_steps

warmup_updates = int(0.05 * total_updates)  # 5% warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_updates,
    num_training_steps=total_updates,
)

# Fast-forward scheduler if resuming
for _ in range(done_updates):
    scheduler.step()

print("Scheduler ready:",
      "total_updates =", total_updates,
      "warmup_updates =", warmup_updates,
      "done_updates =", done_updates)

# ====== MODERN AMP ======
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_val = float(ckpt.get("best_val", float("inf")))
best_path = os.path.join(CKPT_DIR, "BEST.pt")

# ====== TRAIN LOOP ======
t0 = time.time()
step = start_step
optimizer.zero_grad(set_to_none=True)

train_iter = iter(train_loader_100m)

while step < target_steps:
    # get batch (loop forever)
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader_100m)
        batch = next(train_iter)

    x = batch["input_ids"].to(device)
    y = batch["labels"].to(device)

    with torch.amp.autocast("cuda", enabled=use_amp):
        out = model(input_ids=x, labels=y)
        loss = out.loss / grad_accum_steps

    scaler.scale(loss).backward()

    # update every grad_accum_steps
    if (step + 1) % grad_accum_steps == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        scheduler.step()  # <-- THIS is the new big improvement!

    if step % print_every == 0:
        lr_now = scheduler.get_last_lr()[0]
        steps_per_sec = (step - start_step + 1) / (time.time() - t0)
        print(f"[100M built] step {step} | train_loss {(loss.item()*grad_accum_steps):.4f} | lr {lr_now:.2e} | {steps_per_sec:.2f} steps/s")

    # ====== EVAL + SAVE ======
    if step % eval_every == 0 and step > start_step:
        val_loss, val_ppl = eval_built(model, val_loader_100m, eval_iters=eval_iters)
        print(f"[EVAL] step {step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")

        # Save step checkpoint
        ckpt_path = os.path.join(CKPT_DIR, f"built100m_step{step}.pt")
        torch.save({
            "step": step,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val": best_val,
        }, ckpt_path)
        print("Saved:", ckpt_path)

        # Save BEST
        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                "step": step,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
            }, best_path)
            print("✅ New BEST saved:", best_path)

    step += 1

print("Done. Best val loss:", best_val)
print("Best checkpoint:", best_path)

In [ ]:
# ✅ ONE CELL: save model + tokenizer + config + meta to Google Drive

import os, json, torch

# 0) Where to save (clean folder)
EXPORT_DIR = "/content/drive/MyDrive/mini_llm_project/export_100m"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) Save model weights (the actual trained parameters)
MODEL_PATH = os.path.join(EXPORT_DIR, "model_weights.pt")
torch.save(built_model_100m.state_dict(), MODEL_PATH)

# 2) Save tokenizer
TOKENIZER_PATH = os.path.join(EXPORT_DIR, "tokenizer.json")
tokenizer.save(TOKENIZER_PATH)

# 3) Save model config (so you can rebuild same architecture later)
CONFIG_PATH = os.path.join(EXPORT_DIR, "model_config.json")
config = {
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
    "vocab_size": tokenizer.get_vocab_size()
}
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

# 4) Save meta info (best loss, step, name) - ckpt is optional
META_PATH = os.path.join(EXPORT_DIR, "model_meta.json")
meta = {
    "model_name": "MiniGPT-100M-TinyStories",
    "training_step": int(ckpt["step"]) if "ckpt" in globals() and ckpt is not None and "step" in ckpt else None,
    "best_val_loss": float(ckpt["best_val"]) if "ckpt" in globals() and ckpt is not None and "best_val" in ckpt else None
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Saved everything to:", EXPORT_DIR)
print("   -", MODEL_PATH)
print("   -", TOKENIZER_PATH)
print("   -", CONFIG_PATH)
print("   -", META_PATH)

In [ ]:
import os

best_path = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"

print("Exists:", os.path.exists(best_path))

In [ ]:
ckpt = torch.load(best_path, map_location=device)

built_model_100m.load_state_dict(ckpt["model_state"])

built_model_100m.eval()

print("Loaded step:", ckpt["step"])
print("Best val:", ckpt["best_val"])

In [ ]:
# import os, torch, math

# PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
# BEST_PATH = os.path.join(PROJECT_PATH, "checkpoints_built_100m", "BEST.pt")

# ckpt = torch.load(BEST_PATH, map_location=device)

# built_model_100m.load_state_dict(ckpt["model_state"])
# built_model_100m.to(device)
# built_model_100m.eval()

# print("Loaded BEST from:", BEST_PATH)
# print("BEST step:", ckpt.get("step"))
# print("BEST val_loss (stored):", ckpt.get("best_val"))
# if ckpt.get("best_val") is not None:
#     print("BEST val_ppl (computed):", math.exp(ckpt["best_val"]))

In [ ]:
# val_loss, val_ppl = eval_built(built_model_100m, val_loader_100m, eval_iters=200)
# print("Fresh Val Loss:", val_loss)
# print("Fresh Val PPL :", val_ppl)

In [ ]:
import torch.nn.functional as F
import torch

@torch.no_grad()
def generate_built(model, start_text, max_new_tokens=140, temperature=0.9, top_k=50):
    model.eval()
    ids = tokenizer.encode(start_text).ids
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]
        logits = model(input_ids=x_cond).logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

In [ ]:
prompts = [
    "Once upon a time",
    "In a small village",
    "The little robot"
]

for p in prompts:
    print("\n" + "="*60)
    print("PROMPT:", p)
    print(generate_built(built_model_100m, p))

In [ ]:
import torch

def generate_clean(model, tokenizer, prompt,
                   max_new_tokens=140,
                   temperature=0.7,
                   top_p=0.9,
                   top_k=50,
                   repetition_penalty=1.15):
    model.eval()
    device = next(model.parameters()).device

    # tokenizers.Tokenizer -> encode -> ids
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        out_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            pad_token_id=0,     # safe default (we’ll also set EOS below if you have one)
            eos_token_id=None   # keep going until max_new_tokens
        )

    return tokenizer.decode(out_ids[0].tolist())

In [ ]:
prompts = ["Once upon a time", "In a small village", "The little robot"]
for p in prompts:
    print("\n" + "="*60)
    print("PROMPT:", p)
    print(generate_clean(built_model_100m, tokenizer, p))

In [ ]:
v = tokenizer.get_vocab()
for t in ["</s>", "<eos>", "[EOS]", "<|endoftext|>", "<pad>", "[PAD]"]:
    print(t, v.get(t))

In [ ]:
@torch.no_grad()
def generate_strict(model, tokenizer, prompt, max_new_tokens=120):
    model.eval()
    device = next(model.parameters()).device

    vocab = tokenizer.get_vocab()
    pad_id = vocab.get("[PAD]", 0)
    eos_id = vocab.get("[EOS]", None)   # you showed [EOS] = 3

    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,

        # IMPORTANT: turn OFF sampling
        do_sample=False,

        # Beam search (more “safe/clean”)
        num_beams=3,
        early_stopping=True,

        repetition_penalty=1.25,
        no_repeat_ngram_size=4,

        eos_token_id=eos_id,
        pad_token_id=pad_id,
    )

    text = tokenizer.decode(out[0].tolist())
    text = " ".join(text.split())  # basic cleanup only
    return text

In [ ]:
prompts = ["Once upon a time", "In a small village", "The little robot"]
for p in prompts:
    print("\nPROMPT:", p)
    print(generate_strict(built_model_100m, tokenizer, p))

In [ ]:
for i in range(5):
    print("----")
    print(dataset["train"][i]["text"][:500])

In [ ]:
import re
import torch

def clean_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([,.!?;:])", r"\1", s)          # remove space before punctuation
    s = re.sub(r"([,.!?;:])(?=\w)", r"\1 ", s)      # ensure space after punctuation
    s = re.sub(r"(\.\s*){3,}", "... ", s)           # collapse many dots
    s = re.sub(r"\s+'\s*", "'", s)                  # fix apostrophes spacing
    return s.strip()

@torch.no_grad()
def generate_strict(model, tokenizer, prompt,
                    max_new_tokens=120,
                    temperature=0.7,
                    top_k=30,
                    repetition_penalty=1.25,
                    no_repeat_ngram_size=4):
    model.eval()
    device = next(model.parameters()).device

    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    # ✅ Use EOS you actually have: [EOS]
    eos_id = tokenizer.token_to_id("[EOS]")
    if eos_id is None:
        eos_id = None  # if missing, we just won't stop early

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,              # ✅ IMPORTANT: turn OFF sampling (more stable)
        num_beams=3,                  # ✅ beam search helps quality
        early_stopping=True,
        temperature=temperature,      # not used when do_sample=False but ok
        top_k=top_k,                  # not used when do_sample=False but ok
        repetition_penalty=repetition_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
        eos_token_id=eos_id,
        pad_token_id=0
    )

    text = tokenizer.decode(out[0].tolist())
    return clean_text(text)

prompts = ["Once upon a time", "In a small village", "The little robot"]
for p in prompts:
    print("\n" + "="*60)
    print("PROMPT:", p)
    print(generate_strict(built_model_100m, tokenizer, p))

# I need to retrain the model with a god tekonizer

In [ ]:
# ========= OPTION C: New "clean" tokenizer + save to Drive (ONE CELL) =========
!pip -q install tokenizers datasets

import os, json, random
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing

# 0) Paths (clean folder so you won't get confused later)
BASE_DIR = "/content/drive/MyDrive/mini_llm_project"
EXPORT_DIR = os.path.join(BASE_DIR, "artifacts_100m_v2")   # <-- new folder
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) Load dataset (100M_1)
dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")
train_texts = dataset_100m["train"]["text"]

# 2) Train ByteLevel BPE tokenizer (GPT-style)
# vocab_size: 16k-32k is good. Try 16k first (faster).
vocab_size = 16000

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

# Train on a subset for speed (still works well). Increase if you want.
# Using 200k examples is usually enough to learn punctuation/spacing well.
sample_size = min(200_000, len(train_texts))
sample_texts = random.sample(train_texts, sample_size)

tokenizer.train_from_iterator(sample_texts, trainer=trainer)

# Make tokenizer always add BOS/EOS
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")
tokenizer.post_processor = TemplateProcessing(
    single=f"[BOS] $A [EOS]",
    pair=f"[BOS] $A [EOS] $B:1 [EOS]:1",
    special_tokens=[("[BOS]", bos_id), ("[EOS]", eos_id)]
)

pad_id = tokenizer.token_to_id("[PAD]")

# 3) Save tokenizer files to Drive
tokenizer_json_path = os.path.join(EXPORT_DIR, "tokenizer.json")
tokenizer.save(tokenizer_json_path)

tokenizer_meta = {
    "vocab_size": vocab_size,
    "pad_token": "[PAD]",
    "unk_token": "[UNK]",
    "bos_token": "[BOS]",
    "eos_token": "[EOS]",
    "pad_id": pad_id,
    "bos_id": bos_id,
    "eos_id": eos_id,
    "dataset": "eminorhan/tinystories 100M_1"
}
with open(os.path.join(EXPORT_DIR, "tokenizer_meta.json"), "w") as f:
    json.dump(tokenizer_meta, f, indent=2)

print("✅ Saved tokenizer to:", tokenizer_json_path)
print("✅ Saved meta to:", os.path.join(EXPORT_DIR, "tokenizer_meta.json"))

# 4) Quick sanity test: encode/decode should keep punctuation clean
tests = [
    "Once upon a time, there was a robot.",
    "In a small village, Tim's toy broke! He cried.",
    "Hello... why is spacing weird? It shouldn't be."
]
print("\n--- SANITY TEST (encode -> decode) ---")
for t in tests:
    ids = tokenizer.encode(t).ids
    back = tokenizer.decode(ids)
    print("\nIN :", t)
    print("OUT:", back)

In [ ]:
# ========= STEP 2: Retokenize + Chunk to fixed block_size + DataLoaders (ONE CELL) =========
import os, math, random
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer

# 0) Load tokenizer from Drive
EXPORT_DIR = "/content/drive/MyDrive/mini_llm_project/artifacts_100m_v2"
tokenizer = Tokenizer.from_file(os.path.join(EXPORT_DIR, "tokenizer.json"))

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("PAD_ID:", PAD_ID, "BOS_ID:", BOS_ID, "EOS_ID:", EOS_ID)

# 1) Load dataset (100M_1)
dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")

# 2) Tokenize each story -> ids
def tok_batch(batch):
    ids_list = []
    for t in batch["text"]:
        ids = tokenizer.encode(t).ids
        ids_list.append(ids)
    return {"ids": ids_list}

# ⚡ speed tip: you can start with a small subset first to test (then remove select later)
# dataset_100m = dataset_100m.select_columns(["text"])
# dataset_100m["train"] = dataset_100m["train"].select(range(20000))
# dataset_100m["validation"] = dataset_100m["validation"].select(range(2000))

tokd = dataset_100m.map(tok_batch, batched=True, remove_columns=["text"])

# 3) Chunk into fixed blocks for causal LM
block_size = 256  # good default

def group_texts(examples):
    # flatten list of lists
    concatenated = []
    for x in examples["ids"]:
        concatenated.extend(x)

    # keep only full blocks (block_size + 1 for next-token labels)
    total_length = (len(concatenated) // (block_size + 1)) * (block_size + 1)
    concatenated = concatenated[:total_length]

    input_ids = []
    labels = []

    for i in range(0, total_length, block_size + 1):
        chunk = concatenated[i : i + block_size + 1]
        x = chunk[:-1]
        y = chunk[1:]
        input_ids.append(x)
        labels.append(y)

    return {"input_ids": input_ids, "labels": labels}

lm_100m = tokd.map(group_texts, batched=True, remove_columns=["ids"])

print(lm_100m)

# 4) Set format for PyTorch
lm_100m.set_format(type="torch", columns=["input_ids", "labels"])

# 5) DataLoaders (NOW every sample same length => no more "equal size" error)
batch_size = 8  # if OOM, switch to 4

train_loader_100m = DataLoader(lm_100m["train"], batch_size=batch_size, shuffle=True)
val_loader_100m   = DataLoader(lm_100m["validation"], batch_size=batch_size, shuffle=False)

# 6) Quick check: shape should be [B, block_size]
xb = next(iter(train_loader_100m))["input_ids"]
yb = next(iter(train_loader_100m))["labels"]
print("✅ Batch input shape:", xb.shape)
print("✅ Batch label shape:", yb.shape)

In [ ]:
# =========================
# FULL TRAIN + EVAL + SAVE
# =========================
import os, time, math
import torch
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# -------------------------
# 0) REQUIRED THINGS YOU ALREADY HAVE
# -------------------------
# model = built_model_100m
# train_loader_100m, val_loader_100m
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = built_model_100m.to(device)

# -------------------------
# 1) PATHS
# -------------------------
PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
CKPT_DIR = os.path.join(PROJECT_PATH, "checkpoints_built_100m")
os.makedirs(CKPT_DIR, exist_ok=True)

BEST_PATH = os.path.join(CKPT_DIR, "BEST.pt")
LAST_PATH = os.path.join(CKPT_DIR, "LAST.pt")

# -------------------------
# 2) SETTINGS
# -------------------------
target_steps = 50000
print_every = 200
eval_every  = 2000
eval_iters  = 200

lr = 3e-4
weight_decay = 0.1
grad_accum_steps = 4
max_grad_norm = 1.0
use_amp = True

# -------------------------
# 3) OPTIMIZER + AMP
# -------------------------
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type=="cuda"))

# -------------------------
# 4) OPTIONAL: RESUME FROM BEST (if exists)
# -------------------------
start_step = 0
best_val = float("inf")

if os.path.exists(BEST_PATH):
    ckpt = torch.load(BEST_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    start_step = int(ckpt.get("step", 0))
    best_val = float(ckpt.get("best_val", best_val))
    print(f"✅ Loaded BEST checkpoint from {BEST_PATH}")
    print("   step:", start_step, " best_val:", best_val)
else:
    ckpt = {}  # so code doesn't crash later
    print("ℹ️ No BEST checkpoint found. Training from scratch.")

# -------------------------
# 5) SCHEDULER (cosine with warmup)
# NOTE: Scheduler steps only when optimizer.step happens
# -------------------------
total_updates = target_steps // grad_accum_steps
warmup_updates = max(1, int(0.05 * total_updates))  # 5% warmup

done_updates = start_step // grad_accum_steps
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_updates,
    num_training_steps=total_updates
)

# fast-forward scheduler if resuming
for _ in range(done_updates):
    scheduler.step()

print("✅ Scheduler ready:",
      "total_updates=", total_updates,
      "warmup_updates=", warmup_updates,
      "done_updates=", done_updates)

# -------------------------
# 6) EVALUATION FUNCTION
# -------------------------
@torch.no_grad()
def eval_model(model, val_loader, iters=200):
    model.eval()
    losses = []
    val_iter = iter(val_loader)

    for _ in range(iters):
        try:
            batch = next(val_iter)
        except StopIteration:
            val_iter = iter(val_loader)
            batch = next(val_iter)

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        out = model(input_ids=x, labels=y)   # ✅ correct
        loss = out.loss
        losses.append(loss.item())

    model.train()
    avg_loss = sum(losses) / len(losses)
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, ppl

# -------------------------
# 7) TRAIN LOOP
# -------------------------
model.train()
train_iter = iter(train_loader_100m)

t0 = time.time()
running_loss = 0.0

for step in range(start_step, target_steps):
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader_100m)
        batch = next(train_iter)

    x = batch["input_ids"].to(device)
    y = batch["labels"].to(device)

    with torch.cuda.amp.autocast(enabled=(use_amp and device.type=="cuda")):
        out = model(input_ids=x, labels=y)          # ✅ FIXED: keyword args
        loss = out.loss / grad_accum_steps

    scaler.scale(loss).backward()
    running_loss += loss.item()

    # optimizer step every grad_accum_steps
    if (step + 1) % grad_accum_steps == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        # ✅ scheduler AFTER optimizer.step
        scheduler.step()

    # print
    if (step + 1) % print_every == 0:
        elapsed = time.time() - t0
        avg_train_loss = running_loss / print_every
        running_loss = 0.0

        lr_now = optimizer.param_groups[0]["lr"]
        steps_per_sec = print_every / max(elapsed, 1e-9)
        print(f"[100m] step {step+1}/{target_steps} | train_loss {avg_train_loss:.4f} | lr {lr_now:.2e} | {steps_per_sec:.2f} steps/s")
        t0 = time.time()

    # eval + save
    if (step + 1) % eval_every == 0:
        val_loss, val_ppl = eval_model(model, val_loader_100m, iters=eval_iters)
        print(f"   🔎 VAL | step {step+1} | val_loss {val_loss:.4f} | ppl {val_ppl:.2f}")

        # always save LAST
        torch.save({
            "step": step + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val": best_val,
        }, LAST_PATH)
        print("   💾 Saved LAST:", LAST_PATH)

        # save BEST
        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                "step": step + 1,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val": best_val,
            }, BEST_PATH)
            print("   ✅ New BEST saved:", BEST_PATH)

print("✅ Done training.")
print("BEST val loss:", best_val)
print("BEST checkpoint:", BEST_PATH)
print("LAST checkpoint:", LAST_PATH)

In [ ]:
import torch
from tokenizers import Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- paths ----
BEST_PATH = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"
TOKENIZER_PATH = "/content/drive/MyDrive/mini_llm_project/artifacts_100m_v2/tokenizer.json"

# ---- load tokenizer ----
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

# ---- load model ----
ckpt = torch.load(BEST_PATH, map_location=device)

model = built_model_100m.to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

print("Model loaded. Best val loss:", ckpt["best_val"])


# ---- generation function ----
@torch.no_grad()






# ---- generation function ----
@torch.no_grad()
def generate(prompt, max_new_tokens=120, temperature=0.9, top_k=50):

    ids = tokenizer.encode(prompt).ids
    ids = [BOS_ID] + ids

    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):

        logits = model(x)

        next_logits = logits[:, -1, :] / temperature

        v, ix = torch.topk(next_logits, k=top_k)
        probs = torch.softmax(v, dim=-1)

        next_id = ix[:, torch.multinomial(probs, 1)]

        if next_id.item() == EOS_ID:
            break

        x = torch.cat([x, next_id], dim=1)

    text = tokenizer.decode(x[0].tolist())
    return text




In [ ]:
s = "Once upon a time, a brave cat walked into a small village."

enc = tokenizer.encode(s)

print("ids:", enc.ids[:20])
print("decoded:", tokenizer.decode(enc.ids))

In [ ]:
import os, torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Loading BEST checkpoint from:", BEST_PATH)
assert os.path.exists(BEST_PATH), f"BEST_PATH not found: {BEST_PATH}"

ckpt = torch.load(BEST_PATH, map_location=device)

# Some checkpoints store weights directly, others store dict with 'model_state'
if isinstance(ckpt, dict) and "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    # If you saved state_dict directly
    model.load_state_dict(ckpt, strict=True)

model.eval()
print("✅ BEST checkpoint loaded and model set to eval()")

In [ ]:
import torch
import torch.nn.functional as F

def _top_p_filtering(logits, top_p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    probs = F.softmax(sorted_logits, dim=-1)
    cumprobs = torch.cumsum(probs, dim=-1)

    # remove tokens with cumulative probability above threshold
    mask = cumprobs > top_p
    mask[..., 0] = False  # keep at least 1 token
    sorted_logits[mask] = float("-inf")

    # unsort back
    unsorted = torch.full_like(logits, float("-inf"))
    unsorted.scatter_(1, sorted_indices, sorted_logits)
    return unsorted

@torch.no_grad()
def generate_text(
    prompt,
    model,
    tokenizer,
    max_new_tokens=120,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.15,
    eos_token_id=3,
):
    model.eval()
    device = next(model.parameters()).device

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        outputs = model(input_ids)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs[0]

        logits = logits[:, -1, :]  # [1, vocab]

        # repetition penalty (discourage repeating recent tokens)
        if repetition_penalty is not None and repetition_penalty > 1.0:
            recent = input_ids[0, -128:]  # last 128 tokens
            logits[0, recent] = logits[0, recent] / repetition_penalty

        # temperature
        if temperature and temperature > 0:
            logits = logits / temperature

        # top-k
        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(1, ix, v)
            logits = filtered

        # top-p
        if top_p and 0 < top_p < 1:
            logits = _top_p_filtering(logits, top_p=top_p)

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_id], dim=1)

        if next_id.item() == eos_token_id:
            break

    return tokenizer.decode(input_ids[0].tolist())


# QUICK TEST
prompts = [
    "Once upon a time",
    "In a small village",
    "The little robot",
    "One day Tim found",
    "A brave cat",
]

print("\n================ GENERATION TEST ================\n")
for p in prompts:
    print("PROMPT:", p)
    print(generate_text(p, model, tokenizer))
    print("\n" + "-" * 60 + "\n")

#  Today WHere I started

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/content/drive/MyDrive/mini_llm_project/export_100m/tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

EOS_ID = tokenizer.token_to_id("[EOS]")
print("✅ Tokenizer loaded:", TOKENIZER_PATH)
print("vocab_size =", tokenizer.get_vocab_size())
print("EOS_ID =", EOS_ID)

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.get_vocab_size()
block_size = 256
n_embd = 768
n_head = 12
n_layer = 12
dropout = 0.1

cfg = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

model = GPT2LMHeadModel(cfg).to(device)
model.eval()

print("✅ Model created. model vocab =", model.config.vocab_size)

In [ ]:
import os, torch

BEST_PATH = "/content/drive/MyDrive/mini_llm_project/checkpoints_built_100m/BEST.pt"
assert os.path.exists(BEST_PATH), f"BEST checkpoint not found: {BEST_PATH}"

ckpt = torch.load(BEST_PATH, map_location=device)

if isinstance(ckpt, dict) and "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

model.eval()
print("✅ BEST checkpoint loaded:", BEST_PATH)

In [ ]:
s = "Once upon a time, a brave cat walked into a small village."
enc = tokenizer.encode(s)
print("decoded:", tokenizer.decode(enc.ids))  # should look normal

In [ ]:
import torch.nn.functional as F
import torch

@torch.no_grad()
def generate_text(prompt, model, tokenizer, max_new_tokens=120, temperature=0.6, top_k=20, eos_token_id=None):
    model.eval()
    device = next(model.parameters()).device

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        outputs = model(input_ids)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        logits = logits[:, -1, :]

        if temperature and temperature > 0:
            logits = logits / temperature

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(1, ix, v)
            logits = filtered

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

        if eos_token_id is not None and next_id.item() == eos_token_id:
            break

    return tokenizer.decode(input_ids[0].tolist())

In [ ]:
print("EOS_ID =", EOS_ID)
print(generate_text("Once upon a time", model, tokenizer, max_new_tokens=120, temperature=0.6, top_k=20, eos_token_id=EOS_ID))

In [ ]:
print(generate_text(
    "Once upon a time",
    model,
    tokenizer,
    max_new_tokens=80,
    temperature=0.4,
    top_k=10,
    eos_token_id=EOS_ID
))

In [ ]:
prompts = [
    "Once upon a time",
    "There was a little girl named Jenny",
    "Tim and Ben liked to play"
]

for p in prompts:
    print("\nPROMPT:", p)
    print(generate_text(
        p,
        model,
        tokenizer,
        max_new_tokens=80,
        temperature=0.4,
        top_k=10,
        eos_token_id=EOS_ID
    ))
    print("-" * 60)

## What I should do

In [ ]:
import torch
print (torch.__version__)
print (torch.cuda.is_available())
print("GPU Namem: ", torch.cuda.get_device_name(0))

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/mini_llm_project"
CKPT_DIR = f"{PROJECT_PATH}/checkpoints_built_100m"
BEST_PATH = f"{CKPT_DIR}/BEST.pt"
LAST_PATH = f"{CKPT_DIR}/LAST.pt"

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/content/drive/MyDrive/mini_llm_project/export_100m/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

EOS_ID = tokenizer.token_to_id("[EOS]")

print("Tokenizer loaded")
print("vocab size:", tokenizer.get_vocab_size())
print("EOS:", EOS_ID)

In [ ]:
from datasets import load_dataset

dataset_100m = load_dataset("eminorhan/tinystories", "100M_1")
print(dataset_100m)
print(dataset_100m["train"][0]["text"][:200])

In [ ]:
def tokenize_with_eos(examples):
    ids_list = []
    for text in examples["text"]:
        ids = tokenizer.encode(text).ids
        ids.append(EOS_ID)
        ids_list.append(ids)
    return {"ids": ids_list}

tok_100m = dataset_100m.map(
    tokenize_with_eos,
    batched=True,
    remove_columns=["text"]
)

In [ ]:
block_size = 256

def make_blocks(examples):
    input_ids = []
    labels = []

    for ids in examples["ids"]:
        # go through one story at a time
        for i in range(0, len(ids) - 1, block_size):
            chunk = ids[i : i + block_size + 1]

            # need at least 2 tokens to make x and y
            if len(chunk) < 2:
                continue

            x = chunk[:-1]   # input
            y = chunk[1:]    # next-token labels

            input_ids.append(x)
            labels.append(y)

    return {
        "input_ids": input_ids,
        "labels": labels
    }

lm_100m = tok_100m.map(
    make_blocks,
    batched=True,
    remove_columns=["ids"]
)

print("✅ lm_100m created")
print(lm_100m)
print(lm_100m["train"][0].keys())
print("Example input length:", len(lm_100m["train"][0]["input_ids"]))
print("Example label length:", len(lm_100m["train"][0]["labels"]))

In [ ]:
block_size = 256

def make_blocks(examples):
    input_ids = []
    labels = []

    for ids in examples["ids"]:
        # only keep chunks that are big enough for full x/y length 256
        for i in range(0, len(ids) - (block_size + 1) + 1, block_size):
            chunk = ids[i : i + block_size + 1]

            if len(chunk) == block_size + 1:
                x = chunk[:-1]   # length 256
                y = chunk[1:]    # length 256

                input_ids.append(x)
                labels.append(y)

    return {
        "input_ids": input_ids,
        "labels": labels
    }

lm_100m = tok_100m.map(
    make_blocks,
    batched=True,
    remove_columns=["ids"]
)

print("✅ lm_100m created")
print(lm_100m)
print(lm_100m["train"][0].keys())
print("Example input length:", len(lm_100m["train"][0]["input_ids"]))
print("Example label length:", len(lm_100m["train"][0]["labels"]))

In [ ]:
from torch.utils.data import DataLoader
import torch

batch_size = 8

def collate_fn(batch):
    input_ids = torch.tensor([item["input_ids"] for item in batch], dtype=torch.long)
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "labels": labels
    }

train_loader = DataLoader(
    lm_100m["train"],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    lm_100m["validation"],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

print("✅ DataLoaders created")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

batch = next(iter(train_loader))
print("Batch keys:", batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.get_vocab_size()
block_size = 256
n_embd = 768
n_head = 12
n_layer = 12
dropout = 0.1

cfg = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout,
)

model = GPT2LMHeadModel(cfg).to(device)
model.eval()

print("✅ Model created")
print("Using device:", device)
print("Model vocab:", model.config.vocab_size)

In [ ]:
import os, torch

start_step = 0
best_val = float("inf")

if os.path.exists(LAST_PATH):
    print("🔁 Resuming from LAST:", LAST_PATH)
    ckpt = torch.load(LAST_PATH, map_location=device)

    # load model
    if "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"], strict=True)
    elif "model" in ckpt:
        model.load_state_dict(ckpt["model"], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)

    # optional: resume step + best_val if saved
    start_step = int(ckpt.get("step", 0))
    best_val   = float(ckpt.get("best_val", float("inf")))

    print("✅ Resumed. start_step =", start_step, " best_val =", best_val)
else:
    print("🆕 No LAST checkpoint found. Training from scratch.")

In [ ]:
import torch.nn.functional as F

lr = 3e-4
weight_decay = 0.1
max_grad_norm = 1.0
use_amp = True

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

In [ ]:
import torch

@torch.no_grad()
def eval_loss(model, loader, eval_batches=200):
    model.eval()
    losses = []
    for i, batch in enumerate(loader):
        if i >= eval_batches:
            break
        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)
        out = model(input_ids=x, labels=y)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / len(losses)

In [ ]:
target_steps = start_step + 30000
print_every = 200
eval_every = 2000
save_every = 2000   # save LAST often so you don't lose progress
grad_accum_steps = 4

In [ ]:
import math, time
from contextlib import nullcontext

model.train()
t0 = time.time()

step = start_step

train_iter = iter(train_loader)

while step < target_steps:
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    for _ in range(grad_accum_steps):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if (use_amp and device.type=="cuda") else nullcontext()
        with autocast_ctx:
            out = model(input_ids=x, labels=y)
            loss = out.loss / grad_accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item()

    # clip grads
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    scaler.step(optimizer)
    scaler.update()

    step += 1

    if step % print_every == 0:
        dt = time.time() - t0
        print(f"step {step} | train_loss {total_loss:.4f} | time {dt:.1f}s")
        t0 = time.time()

    # eval + save best
    if step % eval_every == 0:
        val = eval_loss(model, val_loader, eval_batches=200)
        print(f"✅ step {step} | val_loss {val:.4f}")

        # save BEST
        if val < best_val:
            best_val = val
            torch.save({
                "model_state": model.state_dict(),
                "step": step,
                "best_val": best_val,
            }, BEST_PATH)
            print("🏆 Saved BEST:", BEST_PATH)

    # always save LAST sometimes
    if step % save_every == 0:
        torch.save({
            "model_state": model.state_dict(),
            "step": step,
            "best_val": best_val,
        }, LAST_PATH)
        print("💾 Saved LAST:", LAST_PATH)

In [ ]:
import torch

ckpt = torch.load(BEST_PATH, map_location=device)

if "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

model.eval()

print("✅ BEST model loaded")
print("BEST step:", ckpt.get("step", "unknown"))
print("BEST val_loss:", ckpt.get("best_val", "unknown"))

In [ ]:
import os
import time
import math
import torch
import torch.nn.functional as F
from contextlib import nullcontext

# =========================
# 1) RESUME FROM LAST
# =========================
start_step = 0
best_val = float("inf")

if os.path.exists(LAST_PATH):
    print("🔁 Resuming from LAST:", LAST_PATH)
    ckpt = torch.load(LAST_PATH, map_location=device)

    if "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"], strict=True)
    elif "model" in ckpt:
        model.load_state_dict(ckpt["model"], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)

    start_step = int(ckpt.get("step", 0))
    best_val = float(ckpt.get("best_val", float("inf")))

    print("✅ Resumed. start_step =", start_step, " best_val =", best_val)
else:
    print("🆕 No LAST checkpoint found. Training from scratch.")

# =========================
# 2) OPTIMIZER + AMP
# =========================
lr = 3e-4
weight_decay = 0.1
max_grad_norm = 1.0
use_amp = True

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

# =========================
# 3) EVAL FUNCTION
# =========================
@torch.no_grad()
def eval_loss(model, loader, eval_batches=200):
    model.eval()
    losses = []

    for i, batch in enumerate(loader):
        if i >= eval_batches:
            break

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        out = model(input_ids=x, labels=y)
        losses.append(out.loss.item())

    model.train()
    return sum(losses) / len(losses)

# =========================
# 4) TRAINING SETTINGS
# =========================
target_steps = 150000
print_every = 200
eval_every = 2000
save_every = 2000
grad_accum_steps = 4

print("🚀 Training from step", start_step, "to", target_steps)

# =========================
# 5) TRAIN LOOP
# =========================
model.train()
t0 = time.time()
step = start_step

train_iter = iter(train_loader)

while step < target_steps:
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0

    for _ in range(grad_accum_steps):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        x = batch["input_ids"].to(device)
        y = batch["labels"].to(device)

        autocast_ctx = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if (use_amp and device.type == "cuda")
            else nullcontext()
        )

        with autocast_ctx:
            out = model(input_ids=x, labels=y)
            loss = out.loss / grad_accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

    scaler.step(optimizer)
    scaler.update()

    step += 1

    if step % print_every == 0:
        dt = time.time() - t0
        print(f"step {step} | train_loss {total_loss:.4f} | time {dt:.1f}s")
        t0 = time.time()

    if step % eval_every == 0:
        val = eval_loss(model, val_loader, eval_batches=200)
        print(f"✅ step {step} | val_loss {val:.4f}")

        if val < best_val:
            best_val = val
            torch.save({
                "model_state": model.state_dict(),
                "step": step,
                "best_val": best_val,
            }, BEST_PATH)
            print("🏆 Saved BEST:", BEST_PATH)

    if step % save_every == 0:
        torch.save({
            "model_state": model.state_dict(),
            "step": step,
            "best_val": best_val,
        }, LAST_PATH)
        print("💾 Saved LAST:", LAST_PATH)

print("🎉 Training finished at step", step)
print("Best val loss:", best_val)

In [ ]:
import torch

ckpt = torch.load(BEST_PATH, map_location=device)

if "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

model.eval()

print("✅ BEST model loaded")
print("BEST step:", ckpt.get("step", "unknown"))
print("BEST val_loss:", ckpt.get("best_val", "unknown"))

In [ ]:



import torch
import torch.nn.functional as F

@torch.no_grad()
def generate_text(prompt, model, tokenizer, max_new_tokens=80, temperature=0.45, top_k=15, eos_token_id=None, repetition_penalty=1.15):

    model.eval()

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):

        outputs = model(input_ids)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        logits = logits[:, -1, :]

        # repetition penalty
        for token in set(input_ids[0].tolist()):
            logits[0, token] /= repetition_penalty

        logits = logits / temperature

        v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
        filtered = torch.full_like(logits, float("-inf"))
        filtered.scatter_(1, ix, v)
        logits = filtered

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return tokenizer.decode(input_ids[0].tolist())
prompts = [
    "Once upon a time",
    "There was a little girl named Jenny",
    "Tim and Ben liked to play"
]

for p in prompts:
    print("\nPROMPT:", p)
    print(generate_text(
        p,
        model,
        tokenizer,
        max_new_tokens=80,
        temperature=0.45,
        top_k=15,
        eos_token_id=EOS_ID,
        repetition_penalty=1.15
    ))
    print("-" * 60)

In [ ]:
# Test the model with Computer Science questions

cs_prompts = [
"Explain what an array is in simple words.",
"What is a variable in programming?",
"Explain what a loop does in a computer program.",
"What is artificial intelligence?",
"Explain recursion in programming like I am a beginner."
]

for p in cs_prompts:
 print("\nPROMPT:", p)


 output = generate_text(
    p,
    model,
    tokenizer,
    max_new_tokens=80,
    temperature=0.45,
    top_k=15,
    eos_token_id=EOS_ID
)

print("AI ANSWER:", output)
print("-" * 70)


# Conclusion

In this project, I successfully built and trained a **GPT-style language model from scratch** using the TinyStories dataset. The model was able to learn basic narrative patterns and generate simple text continuations based on input prompts.

Although the generated text is not always grammatically perfect, the model demonstrates an understanding of story structure and token prediction. This outcome highlights both the potential and the limitations of training language models from scratch with limited computational resources.

This project provided valuable hands-on experience with:

* Transformer-based language model architecture
* Tokenization and vocabulary construction
* Dataset preparation for language modeling
* Training loops and checkpointing strategies
* Text generation techniques

Overall, this project strengthened my understanding of how modern language models work internally and the engineering required to train them. Future improvements could include larger model architectures, longer training time, and fine-tuning on domain-specific datasets to improve fluency and coherence.
